# SAE Train + Evaluate — SUPERVISED backbone
Sirf **supervised** backbone ke liye: SAE train karta hai, purity/entropy + downstream probe + spatial Dice compute karta hai, aur top-10 qualitative figure banata hai. Sab kuch `results/supervised/` ke andar save hota hai. Self-contained, koi variable change nahi karni.

## 1. Imports + Config

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ==================== CONFIG ====================
MODE = "supervised"
METADATA_CSV = r"C:\Users\Raaghav\Desktop\coding\research\brain-mri-ncl-simclr\data\processed_flair\metadata.csv"
CHECKPOINT_DIR = r"C:\Users\Raaghav\Desktop\coding\research\brain-mri-ncl-simclr\checkpoints"
RESULTS_ROOT = r"C:\Users\Raaghav\Desktop\coding\research\brain-mri-ncl-simclr\results"
RESULTS_DIR = os.path.join(RESULTS_ROOT, MODE)   # results/supervised/
IMG_SIZE = 128
BATCH_SIZE = 128
NUM_WORKERS = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# SAE hyperparams
EMBED_DIM = 192
DICT_SIZE = 1536
SAE_K = 24
SAE_EPOCHS = 30
SAE_LR = 1e-3
TOP_K_SLICES = 10

LABEL_NAMES = {0: "non_tumor", 1: "necrotic_core", 2: "edema", 4: "enhancing_tumor"}

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "embeddings"), exist_ok=True)
print(f"Mode: {MODE} | Device: {DEVICE}")

## 2. Dataset + backbone loader

In [ ]:
class BraTSEvalDataset(Dataset):
    def __init__(self, metadata_csv, split):
        df = pd.read_csv(metadata_csv)
        self.df = df[df["split"] == split].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        data = np.load(row["file_path"], allow_pickle=True).item()
        image = torch.from_numpy(data["image"]).unsqueeze(0).float()
        return image, int(row["dominant_label"]), int(row["has_tumor"]), idx


def build_vit_tiny_backbone(img_size=IMG_SIZE, in_chans=1):
    return timm.create_model(
        "vit_tiny_patch16_224", pretrained=False,
        img_size=img_size, in_chans=in_chans, num_classes=0,
    )


def load_backbone(mode):
    backbone = build_vit_tiny_backbone().to(DEVICE)
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"{mode}_backbone.pt")
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    backbone.load_state_dict(ckpt["backbone_state_dict"])
    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad = False
    return backbone

print("Dataset + backbone loader ready.")

## 3. Extract representations (train + val), cached

In [ ]:
def extract_representations(mode, split):
    cache_path = os.path.join(RESULTS_DIR, "embeddings", f"{split}_embeddings.npz")

    if os.path.exists(cache_path):
        d = np.load(cache_path)
        return d["embeddings"], d["labels"], d["has_tumor"], d["indices"]

    backbone = load_backbone(mode)
    ds = BraTSEvalDataset(METADATA_CSV, split)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    all_embeds, all_labels, all_tumor, all_idx = [], [], [], []
    with torch.no_grad():
        for images, labels, has_tumor, idx in tqdm(loader, desc=f"Extracting {mode}/{split}"):
            images = images.to(DEVICE)
            with torch.amp.autocast(DEVICE):
                embeds = backbone(images)
            all_embeds.append(embeds.float().cpu().numpy())
            all_labels.append(labels.numpy())
            all_tumor.append(has_tumor.numpy())
            all_idx.append(idx.numpy())

    embeddings = np.concatenate(all_embeds, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    has_tumor = np.concatenate(all_tumor, axis=0)
    indices = np.concatenate(all_idx, axis=0)

    np.savez(cache_path, embeddings=embeddings, labels=labels, has_tumor=has_tumor, indices=indices)
    del backbone
    torch.cuda.empty_cache()
    return embeddings, labels, has_tumor, indices

print("Extraction ready.")

## 4. TopK Sparse Autoencoder

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, input_dim=EMBED_DIM, dict_size=DICT_SIZE, k=SAE_K):
        super().__init__()
        self.k = k
        self.pre_bias = nn.Parameter(torch.zeros(input_dim))
        self.encoder = nn.Linear(input_dim, dict_size)
        self.decoder = nn.Linear(dict_size, input_dim, bias=False)

    def forward(self, x):
        x_centered = x - self.pre_bias
        acts = F.relu(self.encoder(x_centered))
        topk_vals, topk_idx = torch.topk(acts, self.k, dim=1)
        sparse_acts = torch.zeros_like(acts).scatter_(1, topk_idx, topk_vals)
        recon = self.decoder(sparse_acts) + self.pre_bias
        return recon, sparse_acts


def train_sae(embeddings, epochs=SAE_EPOCHS):
    X = torch.from_numpy(embeddings).float()
    dataset = torch.utils.data.TensorDataset(X)
    loader = DataLoader(dataset, batch_size=512, shuffle=True)

    sae = TopKSAE().to(DEVICE)
    optimizer = torch.optim.Adam(sae.parameters(), lr=SAE_LR)

    losses = []
    for epoch in tqdm(range(epochs), desc="SAE training"):
        total_loss = 0.0
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            recon, _ = sae(batch)
            loss = F.mse_loss(recon, batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        losses.append(total_loss / len(loader))

    return sae, losses

print("SAE ready.")

## 5. Purity/Entropy metric (Section 5.1)

In [ ]:
def compute_purity_entropy(sae, embeddings, labels, top_k=TOP_K_SLICES):
    X = torch.from_numpy(embeddings).float().to(DEVICE)
    with torch.no_grad():
        _, sparse_acts = sae(X)
    sparse_acts = sparse_acts.cpu().numpy()

    alive_mask = (sparse_acts > 0).any(axis=0)
    alive_features = np.where(alive_mask)[0]

    purities, entropies = [], []
    for feat_idx in alive_features:
        acts_col = sparse_acts[:, feat_idx]
        n_active = (acts_col > 0).sum()
        if n_active < 1:
            continue
        k = min(top_k, n_active)
        top_indices = np.argsort(acts_col)[-k:]
        top_labels = labels[top_indices]

        vals, counts = np.unique(top_labels, return_counts=True)
        purity = counts.max() / k
        probs = counts / k
        entropy = -np.sum(probs * np.log(probs + 1e-12))

        purities.append(purity)
        entropies.append(entropy)

    return {
        "n_alive_features": int(len(alive_features)),
        "mean_purity": float(np.mean(purities)) if purities else 0.0,
        "mean_entropy": float(np.mean(entropies)) if entropies else 0.0,
    }, sparse_acts, alive_features

print("Metric ready.")

## 6. Downstream linear probe

In [ ]:
def downstream_probe_accuracy(train_embeds, train_tumor, val_embeds, val_tumor):
    clf = LogisticRegression(max_iter=1000)
    clf.fit(train_embeds, train_tumor)
    preds = clf.predict(val_embeds)
    return float(accuracy_score(val_tumor, preds))

print("Probe ready.")

## 7. Run: extract -> train SAE -> metrics

In [ ]:
print(f"{'='*50}\n{MODE.upper()}\n{'='*50}")

train_embeds, train_labels, train_tumor, _ = extract_representations(MODE, "train")
val_embeds, val_labels, val_tumor, val_idx = extract_representations(MODE, "val")

print(f"Training SAE for {MODE}...")
sae, sae_losses = train_sae(train_embeds)

metrics, sparse_acts, alive_features = compute_purity_entropy(sae, val_embeds, val_labels)
probe_acc = downstream_probe_accuracy(train_embeds, train_tumor, val_embeds, val_tumor)

print(f"\n{MODE}: purity={metrics['mean_purity']:.3f} | entropy={metrics['mean_entropy']:.3f} | "
      f"alive_features={metrics['n_alive_features']} | probe_acc={probe_acc:.3f}")

# SAE checkpoint save karo
os.makedirs(os.path.join(CHECKPOINT_DIR, "sae"), exist_ok=True)
torch.save(sae.state_dict(), os.path.join(CHECKPOINT_DIR, "sae", f"{MODE}_sae.pt"))

# SAE loss curve save karo
fig_sae = plt.figure(figsize=(6, 3.5))
plt.plot(sae_losses, marker="o")
plt.title(f"{MODE.upper()} SAE Reconstruction Loss"); plt.xlabel("Epoch")
plt.tight_layout()
fig_sae.savefig(os.path.join(RESULTS_DIR, "figures", "sae_loss_curve.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Qualitative figure — best feature ke top-10 slices

In [ ]:
def get_best_feature_top_slices(sae, val_embeds, val_labels, top_k=TOP_K_SLICES):
    X = torch.from_numpy(val_embeds).float().to(DEVICE)
    with torch.no_grad():
        _, sparse_acts = sae(X)
    sparse_acts = sparse_acts.cpu().numpy()

    alive_mask = (sparse_acts > 0).any(axis=0)
    alive_features = np.where(alive_mask)[0]

    best_purity, best_feat, best_top_indices = -1, None, None
    for feat_idx in alive_features:
        acts_col = sparse_acts[:, feat_idx]
        n_active = (acts_col > 0).sum()
        if n_active < top_k:
            continue
        top_local_idx = np.argsort(acts_col)[-top_k:]
        top_labels = val_labels[top_local_idx]
        vals, counts = np.unique(top_labels, return_counts=True)
        purity = counts.max() / top_k
        if purity > best_purity:
            best_purity = purity
            best_feat = feat_idx
            best_top_indices = top_local_idx

    return best_feat, best_top_indices, best_purity


best_feat, top_indices, purity = get_best_feature_top_slices(sae, val_embeds, val_labels)

df_val = pd.read_csv(METADATA_CSV)
df_val = df_val[df_val["split"] == "val"].reset_index(drop=True)

fig, axes = plt.subplots(1, TOP_K_SLICES, figsize=(2 * TOP_K_SLICES, 2.5))
for col, local_idx in enumerate(top_indices):
    real_idx = val_idx[local_idx]
    row_data = df_val.iloc[real_idx]
    data = np.load(row_data["file_path"], allow_pickle=True).item()
    axes[col].imshow(data["image"], cmap="gray")
    axes[col].axis("off")

plt.suptitle(f"{MODE.upper()} — Feature #{best_feat} Top-10 (purity={purity:.2f})", fontsize=12)
plt.tight_layout()

fig_path = os.path.join(RESULTS_DIR, "figures", f"top10_{MODE}.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

# Baad mein cross-backbone comparison figure banane ke liye, indices bhi save kar do
np.savez(os.path.join(RESULTS_DIR, "best_feature_info.npz"),
         best_feat=best_feat, top_indices=top_indices, val_idx=val_idx, purity=purity)

print(f"Figure saved: {fig_path}")

## 9. Spatial Dice metric (Section 5.2, optional)

In [ ]:
PATCH_SIZE = 16
GRID_SIZE = IMG_SIZE // PATCH_SIZE


def extract_patch_tokens(backbone, images):
    with torch.no_grad():
        tokens = backbone.forward_features(images)
    return tokens[:, 1:, :]  # CLS token drop


def patch_dice_for_feature(mode, sae, feat_idx, n_samples=30, activation_threshold_pct=75):
    backbone = load_backbone(mode)
    df_val_local = pd.read_csv(METADATA_CSV)
    df_val_local = df_val_local[(df_val_local["split"] == "val") & (df_val_local["has_tumor"] == 1)].reset_index(drop=True)

    n_samples = min(n_samples, len(df_val_local))
    sample_rows = df_val_local.sample(n=n_samples, random_state=42)

    dice_scores = []
    for _, row in sample_rows.iterrows():
        data = np.load(row["file_path"], allow_pickle=True).item()
        image = torch.from_numpy(data["image"]).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
        seg_mask = data["seg_mask"]

        patch_tokens = extract_patch_tokens(backbone, image).squeeze(0)

        with torch.no_grad():
            _, sparse_acts = sae(patch_tokens)
        feat_acts = sparse_acts[:, feat_idx].cpu().numpy()

        if feat_acts.max() == 0:
            continue

        threshold = np.percentile(feat_acts, activation_threshold_pct)
        high_act_mask = (feat_acts > threshold).reshape(GRID_SIZE, GRID_SIZE)

        tumor_binary = (seg_mask > 0).astype(np.float32)
        gt_patch_mask = np.zeros((GRID_SIZE, GRID_SIZE), dtype=bool)
        for i in range(GRID_SIZE):
            for j in range(GRID_SIZE):
                patch = tumor_binary[i*PATCH_SIZE:(i+1)*PATCH_SIZE, j*PATCH_SIZE:(j+1)*PATCH_SIZE]
                gt_patch_mask[i, j] = patch.mean() > 0.1

        intersection = np.logical_and(high_act_mask, gt_patch_mask).sum()
        dice = (2 * intersection) / (high_act_mask.sum() + gt_patch_mask.sum() + 1e-8)
        dice_scores.append(dice)

    del backbone
    torch.cuda.empty_cache()
    return float(np.mean(dice_scores)) if dice_scores else 0.0, len(dice_scores)


print(f"Computing spatial Dice for {MODE} (feature #{best_feat})...")
mean_dice, n_valid = patch_dice_for_feature(MODE, sae, best_feat)
print(f"{MODE}: mean_dice={mean_dice:.3f} (n={n_valid} samples)")

## 10. Save results — is mode ka metrics.json + shared final_results_table.json update

In [ ]:
mode_results = {
    "sae_final_loss": sae_losses[-1],
    "n_alive_features": metrics["n_alive_features"],
    "mean_purity": metrics["mean_purity"],
    "mean_entropy": metrics["mean_entropy"],
    "downstream_probe_accuracy": probe_acc,
    "spatial_dice": mean_dice,
    "best_feature_idx": int(best_feat),
    "best_feature_purity": float(purity),
}

# 1. Is mode ka apna metrics.json
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(mode_results, f, indent=2)

# 2. Shared final_results_table.json update (read-merge-write, teeno notebooks isi file mein likhengi)
final_table_path = os.path.join(RESULTS_ROOT, "final_results_table.json")
final_table = {}
if os.path.exists(final_table_path):
    with open(final_table_path) as f:
        final_table = json.load(f)
final_table[MODE] = mode_results
with open(final_table_path, "w") as f:
    json.dump(final_table, f, indent=2)

print(f"Saved: {os.path.join(RESULTS_DIR, 'metrics.json')}")
print(f"Updated: {final_table_path}")
print(json.dumps(mode_results, indent=2))